In [1]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/home/samuel-logan/uni/ENGG2112/lake-erie-forecast")
DATA_FOLDER = PROJECT / "data" / "raw" / "ai_2023"

# Search inside the extracted folder, including any subfolders.
matches = list(DATA_FOLDER.rglob("*mmc3.xlsx"))

if len(matches) != 1:
    raise FileNotFoundError(
        f"Expected one mmc3.xlsx file in {DATA_FOLDER}, "
        f"but found {len(matches)}."
    )

# The second Excel row contains the column names.
bloom_data = pd.read_excel(
    matches[0],
    sheet_name="all",
    header=1,
)

# Translate the authors' classes into our binary target.
bloom_data["significant_bloom"] = bloom_data["2-level"].map({
    1: 0,  # Light/mild
    2: 1,  # Significant/extreme
})

print("File:", matches[0])
print("Observations:", len(bloom_data))
print("Years:", bloom_data["year"].min(), "to", bloom_data["year"].max())
print("\nClass counts:")
print(bloom_data["significant_bloom"].value_counts().sort_index())

display(
    bloom_data[
        ["year", "period", "CI", "2-level", "significant_bloom"]
    ].head(10)
)

File: /home/samuel-logan/uni/ENGG2112/lake-erie-forecast/data/raw/ai_2023/1-s2.0-S0043135423001458-mmc3.xlsx
Observations: 240
Years: 2002 to 2019

Class counts:
significant_bloom
0    185
1     55
Name: count, dtype: int64


,year,period,CI,2-level,significant_bloom
0,2002,601-610,1.649855,1,0
1,2002,611-620,1.722002,1,0
2,2002,621-630,2.208485,1,0
3,2002,701-710,2.108872,1,0
4,2002,711-720,2.702665,1,0
5,2002,721-730,2.539468,1,0
6,2002,731-809,2.791853,1,0
7,2002,810-819,2.573102,1,0
8,2002,820-829,3.150023,1,0
9,2002,830-908,2.876797,1,0


In [3]:
# Separate each period into its start and end codes.
period_parts = bloom_data["period"].astype(str).str.strip().str.split(
    "-", expand=True
)

def parse_period_date(years, codes):
    # Pad codes: "601" becomes "0601" (June 1).
    codes = codes.str.strip().str.zfill(4)

    return pd.to_datetime({
        "year": years.astype(int),
        "month": codes.str[:2].astype(int),
        "day": codes.str[2:].astype(int),
    })

bloom_data["period_start"] = parse_period_date(
    bloom_data["year"], period_parts[0]
)

bloom_data["period_end"] = parse_period_date(
    bloom_data["year"], period_parts[1]
)

bloom_data = bloom_data.sort_values("period_start").reset_index(drop=True)

# Inclusive duration: June 1–10 contains 10 days.
bloom_data["duration_days"] = (
    bloom_data["period_end"] - bloom_data["period_start"]
).dt.days + 1

# Count unobserved days between consecutive records within each year.
previous_end = bloom_data.groupby("year")["period_end"].shift(1)

bloom_data["gap_before_days"] = (
    bloom_data["period_start"] - previous_end
).dt.days - 1

print("Period lengths:")
print(bloom_data["duration_days"].value_counts().sort_index())

print("\nRecords with a gap before them, within the same year:")
display(
    bloom_data.loc[
        bloom_data["gap_before_days"] > 0,
        ["year", "period_start", "period_end", "gap_before_days"]
    ]
)

print("\nClass counts by year:")
display(
    bloom_data.groupby("year").agg(
        total_periods=("significant_bloom", "size"),
        significant_periods=("significant_bloom", "sum"),
    )
)

Period lengths:
duration_days
10    221
11      9
13     10
Name: count, dtype: int64

Records with a gap before them, within the same year:


,year,period_start,period_end,gap_before_days
63,2006,2006-07-21,2006-07-30,10.0
64,2006,2006-08-10,2006-08-19,10.0
65,2006,2006-09-09,2006-09-18,20.0



Class counts by year:


,total_periods,significant_periods
year,,
2002,15,0
2003,15,1
2004,15,0
2005,15,0
2006,9,0
2007,15,0
2008,15,2
2009,15,3
2010,14,2


In [4]:
bloom_data["split"] = "train"

bloom_data.loc[
    bloom_data["year"].between(2015, 2016), "split"
] = "validation"

bloom_data.loc[
    bloom_data["year"].between(2017, 2019), "split"
] = "test"

display(
    bloom_data.groupby("split").agg(
        periods=("significant_bloom", "size"),
        significant_periods=("significant_bloom", "sum"),
    ).reindex(["train", "validation", "test"])
)

,periods,significant_periods
split,,
train,187,32
validation,22,13
test,31,10


In [5]:
daily_matches = list(DATA_FOLDER.rglob("*mmc1.xlsx"))

if len(daily_matches) != 1:
    raise FileNotFoundError(
        f"Expected one mmc1.xlsx file, found {len(daily_matches)}."
    )

environment_daily = pd.read_excel(
    daily_matches[0],
    sheet_name="daily data",
)

print("Rows:", len(environment_daily))

print("\nColumn names:")
for column in environment_daily.columns:
    print(repr(column))

print("\nFirst five rows:")
display(environment_daily.head())

print("\nMissing values per column:")
display(environment_daily.isna().sum())

Rows: 7666

Column names:
'Unnamed: 0'
'Datetime (date and time of sample collection)'
'Flow, CFS'
'Conductivity, μmho'
'Flow volume, m3'
'TP, m.ton as P'
'SRP, m.ton as P'
'SS,\nm.ton'
'NO23, m.ton as N'
'TKN, m.ton (Total Kjeldahl nitrogen)'
'Chloride, m.ton'
'Sulfate, m.ton'
'Silica, m.ton'
'PREC,\nin'
'Solar radiation, W/m2'
'wind speed, m/s'
'Water temperature, ℃'
'Water level near the Maumee River， m'
'Water level near the Detroit River, m'
'ICE COVER, %'
' Air TEMP'

First five rows:


,Unnamed: 0,Datetime (date and time of sample collection),"Flow, CFS","Conductivity, μmho","Flow volume, m3","TP, m.ton as P","SRP, m.ton as P","SS,\nm.ton","NO23, m.ton as N","TKN, m.ton (Total Kjeldahl nitrogen)",...,"Sulfate, m.ton","Silica, m.ton","PREC,\nin","Solar radiation, W/m2","wind speed, m/s","Water temperature, ℃",Water level near the Maumee River， m,"Water level near the Detroit River, m","ICE COVER, %",Air TEMP
0,1.0,2000-01-01 20:00:00,520.0,961.000000,1271462.40,0.107057,0.028481,9.790260,4.399260,1.166440,...,173.046033,4.437404,0.0,28.833333,3.775000,0.333333,173.856,173.945,8.8,1.291667
1,2.0,2000-01-02 20:00:00,500.0,965.000000,1222560.00,0.112598,0.029708,7.702128,4.364539,1.187350,...,168.591024,4.792435,4.6,24.833333,5.420833,6.145833,173.781,173.913,9.3,7.250000
2,3.0,2000-01-03 20:00:00,700.0,956.606003,1711584.00,0.163249,0.041477,11.761112,6.236877,1.681303,...,240.502269,6.635711,7.9,20.083333,4.747917,3.520833,174.056,174.059,9.8,3.291667
3,4.0,2000-01-04 20:00:00,908.0,910.000000,2220168.96,0.245773,0.041073,23.447204,7.370961,2.897765,...,291.508184,8.325634,0.5,36.854167,7.550000,3.937500,173.527,173.729,10.3,2.250000
4,5.0,2000-01-05 20:00:00,1070.0,907.000000,2616278.40,0.318139,0.041860,27.209295,11.040695,3.053459,...,339.592936,11.563951,0.0,30.333333,3.752083,-1.479167,173.819,173.914,8.8,-1.666667



Missing values per column:


Unnamed: 0                                       184
Datetime (date and time of sample collection)      0
Flow, CFS                                        184
Conductivity, μmho                               186
Flow volume, m3                                  184
TP, m.ton as P                                   184
SRP, m.ton as P                                  184
SS,\nm.ton                                       184
NO23, m.ton as N                                 184
TKN, m.ton (Total Kjeldahl nitrogen)             184
Chloride, m.ton                                  184
Sulfate, m.ton                                   184
Silica, m.ton                                    184
PREC,\nin                                        366
Solar radiation, W/m2                            731
wind speed, m/s                                  731
Water temperature, ℃                             731
Water level near the Maumee River， m               0
Water level near the Detroit River, m         

In [6]:
# Make a separate working copy with simpler column names.
environment = environment_daily.rename(columns={
    "Datetime (date and time of sample collection)": "date",
    "Flow, CFS": "flow",
    "PREC,\nin": "rainfall",
    "Solar radiation, W/m2": "solar",
    "wind speed, m/s": "wind",
    "Water level near the Maumee River， m": "level_maumee",
    "Water level near the Detroit River, m": "level_detroit",
}).copy()

# Treat each timestamp as its recorded calendar date.
environment["date"] = pd.to_datetime(
    environment["date"], errors="raise"
).dt.normalize()

environment = environment.sort_values("date")

environment_columns = [
    "flow", "rainfall", "solar", "wind",
    "level_maumee", "level_detroit",
]

print("First date:", environment["date"].min())
print("Last date:", environment["date"].max())
print("Repeated dates:", environment["date"].duplicated().sum())

# Check whether entire calendar dates are absent.
expected_dates = pd.date_range(
    environment["date"].min(),
    environment["date"].max(),
    freq="D",
)

missing_dates = expected_dates.difference(environment["date"])
print("Missing calendar dates:", len(missing_dates))

# Check missing measurements during the years used for modelling.
relevant = environment.loc[
    environment["date"].dt.year.between(2002, 2019)
].copy()

missing_by_year = (
    relevant.set_index("date")[environment_columns]
    .isna()
    .groupby(relevant.set_index("date").index.year)
    .sum()
)

print("\nMissing measurements by year:")
display(missing_by_year)

# Inspect value ranges using training years only.
training_environment = relevant.loc[
    relevant["date"].dt.year <= 2014,
    environment_columns,
]

print("\nTraining-year measurement ranges:")
display(
    training_environment.describe(
        percentiles=[0.01, 0.5, 0.99]
    ).T
)

First date: 2000-01-01 00:00:00
Last date: 2020-12-31 00:00:00
Repeated dates: 0
Missing calendar dates: 5

Missing measurements by year:


,flow,rainfall,solar,wind,level_maumee,level_detroit
date,,,,,,
2002,0,0,0,0,0,0
2003,0,0,0,0,0,0
2004,0,0,0,0,0,0
2005,0,0,0,0,0,0
2006,0,0,0,0,0,0
2007,0,0,0,0,0,0
2008,0,0,0,0,0,0
2009,0,0,0,0,0,0
2010,0,0,0,0,0,0



Training-year measurement ranges:


,count,mean,std,min,1%,50%,99%,max
flow,4745.0,6447.488303,10243.674690,54.600000,149.440000,2240.000000,48936.000000,90100.000000
rainfall,4745.0,2.524847,6.715401,0.000000,0.000000,0.000000,33.668000,86.100000
solar,4745.0,69.086670,34.402673,6.166667,15.485833,62.500000,146.697500,169.916667
wind,4745.0,4.071577,1.792161,0.706250,1.225000,3.760417,9.402333,14.400000
level_maumee,4745.0,174.124176,0.256791,172.678000,173.416000,174.158000,174.634120,174.940000
level_detroit,4745.0,174.214231,0.230268,173.121000,173.661320,174.246000,174.624560,174.762000


In [7]:
print("Missing calendar dates:")
for date in missing_dates:
    print(date.date())

# Insert absent dates as rows containing missing values.
# This does not fill or estimate any measurements.
environment_calendar = (
    environment.set_index("date")[environment_columns]
    .reindex(expected_dates)
)

environment_calendar.index.name = "date"

# Count available readings in the ten days before each target period.
coverage_rows = []

for _, row in bloom_data.iterrows():
    start = row["period_start"]

    input_dates = pd.date_range(
        start - pd.Timedelta(days=10),
        start - pd.Timedelta(days=1),
        freq="D",
    )

    counts = environment_calendar.reindex(input_dates).count()

    coverage_rows.append({
        "year": row["year"],
        "period": row["period"],
        "split": row["split"],
        **counts.to_dict(),
    })

input_coverage = pd.DataFrame(coverage_rows)

print("\nForecast periods with fewer than 10 readings for any input:")
display(
    input_coverage.loc[
        input_coverage[environment_columns].lt(10).any(axis=1)
    ]
)

Missing calendar dates:
2000-02-29
2004-02-29
2008-02-29
2012-02-29
2016-02-29

Forecast periods with fewer than 10 readings for any input:


,year,period,split,flow,rainfall,solar,wind,level_maumee,level_detroit
231,2019,701-710,test,10,10,0,0,10,10
232,2019,711-720,test,10,10,0,0,10,10
233,2019,721-731,test,10,10,0,0,10,10
234,2019,801-810,test,10,10,0,0,10,10
235,2019,811-820,test,10,10,0,0,10,10
236,2019,821-831,test,10,10,0,0,10,10
237,2019,901-910,test,10,10,0,0,10,10
238,2019,911-920,test,10,10,0,0,10,10
239,2019,921-930,test,10,10,0,0,10,10


In [8]:
feature_rows = []

required_columns = [
    "flow", "rainfall", "level_maumee", "level_detroit"
]

for _, row in bloom_data.iterrows():
    forecast_start = row["period_start"]

    input_start = forecast_start - pd.Timedelta(days=10)
    input_end = forecast_start - pd.Timedelta(days=1)

    input_dates = pd.date_range(input_start, input_end, freq="D")

    history = environment_calendar.reindex(
        input_dates
    )[required_columns]

    # Stop rather than silently calculate from incomplete data.
    if not history.count().eq(10).all():
        raise ValueError(
            f"Incomplete input window for {forecast_start.date()}"
        )

    feature_rows.append({
        "period_start": forecast_start,
        "period_end": row["period_end"],
        "input_start": input_start,
        "input_end": input_end,
        "split": row["split"],

        # Model inputs: preceding ten calendar days.
        "flow_mean_10d": history["flow"].mean(),
        "rainfall_total_10d": history["rainfall"].sum(),
        "maumee_level_mean_10d": history["level_maumee"].mean(),
        "detroit_level_mean_10d": history["level_detroit"].mean(),

        # Known in advance.
        "day_of_year": forecast_start.dayofyear,

        # Outcome the model will learn to predict.
        "significant_bloom": int(row["significant_bloom"]),
    })

forecast_data = pd.DataFrame(feature_rows)

feature_columns = [
    "flow_mean_10d",
    "rainfall_total_10d",
    "maumee_level_mean_10d",
    "detroit_level_mean_10d",
    "day_of_year",
]

# Check that every input window ends before its target period.
assert (
    forecast_data["input_end"] < forecast_data["period_start"]
).all()

assert forecast_data[feature_columns].notna().all().all()

print("Forecast examples:", len(forecast_data))
print("Number of model inputs:", len(feature_columns))

display(forecast_data.head())

display(
    forecast_data.groupby("split").agg(
        examples=("significant_bloom", "size"),
        significant_periods=("significant_bloom", "sum"),
    ).reindex(["train", "validation", "test"])
)

Forecast examples: 240
Number of model inputs: 5


,period_start,period_end,input_start,input_end,split,flow_mean_10d,rainfall_total_10d,maumee_level_mean_10d,detroit_level_mean_10d,day_of_year,significant_bloom
0,2002-06-01,2002-06-10,2002-05-22,2002-05-31,train,4421.0,9.4,174.3540,174.4161,152,0
1,2002-06-11,2002-06-20,2002-06-01,2002-06-10,train,3826.0,39.4,174.4007,174.4379,162,0
2,2002-06-21,2002-06-30,2002-06-11,2002-06-20,train,1500.0,2.0,174.3137,174.3977,172,0
3,2002-07-01,2002-07-10,2002-06-21,2002-06-30,train,2164.3,9.4,174.3059,174.4116,182,0
4,2002-07-11,2002-07-20,2002-07-01,2002-07-10,train,787.1,2.0,174.3127,174.4167,192,0


,examples,significant_periods
split,,
train,187,32
validation,22,13
test,31,10


In [9]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# Separate inputs (X) from the actual answers (y).
train_rows = forecast_data["split"].eq("train")
validation_rows = forecast_data["split"].eq("validation")

X_train = forecast_data.loc[train_rows, feature_columns]
y_train = forecast_data.loc[train_rows, "significant_bloom"]

X_validation = forecast_data.loc[validation_rows, feature_columns]
y_validation = forecast_data.loc[validation_rows, "significant_bloom"]

# Baseline: always predict the most common training class.
baseline_classifier = DummyClassifier(strategy="most_frequent")

# Our agreed starting model.
rf_classifier = RandomForestClassifier(
    n_estimators=200,
    max_depth=4,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

# Learn using training data only.
baseline_classifier.fit(X_train, y_train)
rf_classifier.fit(X_train, y_train)

# Make validation predictions without supplying the actual answers.
baseline_predictions = baseline_classifier.predict(X_validation)

positive_class_index = list(rf_classifier.classes_).index(1)
rf_validation_scores = rf_classifier.predict_proba(
    X_validation
)[:, positive_class_index]

rf_predictions = (rf_validation_scores >= 0.5).astype(int)

# Compare predictions with the actual validation answers.
def classification_metrics(name, predictions):
    return {
        "approach": name,
        "examples": len(y_validation),
        "accuracy": accuracy_score(y_validation, predictions),
        "precision": precision_score(
            y_validation, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_validation, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_validation, predictions, zero_division=0
        ),
    }

classification_validation_results = pd.DataFrame([
    classification_metrics("Majority baseline", baseline_predictions),
    classification_metrics("Random forest", rf_predictions),
])

display(classification_validation_results.round(3))

for name, predictions in [
    ("Majority baseline", baseline_predictions),
    ("Random forest", rf_predictions),
]:
    print(f"\n{name}:")
    display(pd.DataFrame(
        confusion_matrix(y_validation, predictions, labels=[0, 1]),
        index=["Actual: lower severity", "Actual: significant"],
        columns=["Predicted: lower severity", "Predicted: significant"],
    ))

,approach,examples,accuracy,precision,recall,F1
0,Majority baseline,22,0.409,0.000,0.000,0.00
1,Random forest,22,0.591,0.611,0.846,0.71



Majority baseline:


,Predicted: lower severity,Predicted: significant
Actual: lower severity,9,0
Actual: significant,13,0



Random forest:


,Predicted: lower severity,Predicted: significant
Actual: lower severity,2,7
Actual: significant,2,11


In [10]:
validation_details = forecast_data.loc[
    validation_rows,
    ["period_start", "period_end", "significant_bloom"],
].copy()

validation_details["model_score"] = rf_validation_scores
validation_details["prediction"] = rf_predictions

validation_details["outcome"] = [
    "Correct warning" if actual == 1 and predicted == 1 else
    "Missed significant period" if actual == 1 else
    "False alarm" if predicted == 1 else
    "Correct lower severity"
    for actual, predicted in zip(y_validation, rf_predictions)
]

display(validation_details.round({"model_score": 3}))

,period_start,period_end,significant_bloom,model_score,prediction,outcome
187,2015-07-11,2015-07-20,1,0.090,0,Missed significant period
188,2015-07-21,2015-07-30,1,0.494,0,Missed significant period
189,2015-07-31,2015-08-09,1,0.591,1,Correct warning
190,2015-08-10,2015-08-19,1,0.841,1,Correct warning
191,2015-08-20,2015-08-29,1,0.836,1,Correct warning
192,2015-08-30,2015-09-08,1,0.870,1,Correct warning
193,2015-09-09,2015-09-18,1,0.870,1,Correct warning
194,2015-09-19,2015-09-28,1,0.857,1,Correct warning
195,2015-09-29,2015-10-08,1,0.863,1,Correct warning
196,2015-10-09,2015-10-18,0,0.823,1,False alarm


In [11]:
# Historical bloom observations, indexed by their ending date.
previous_observations = bloom_data[
    ["period_end", "CI", "significant_bloom"]
].rename(columns={
    "period_end": "previous_period_end",
    "CI": "previous_CI",
    "significant_bloom": "previous_bloom_class",
})

forecast_with_history = forecast_data.copy()

# Only accept a previous period ending the day before this one starts.
forecast_with_history["previous_period_end"] = (
    forecast_with_history["period_start"] - pd.Timedelta(days=1)
)

forecast_with_history = forecast_with_history.merge(
    previous_observations,
    on="previous_period_end",
    how="left",
    validate="many_to_one",
)

forecast_with_history["has_previous_CI"] = (
    forecast_with_history["previous_CI"].notna()
)

print("Previous CI availability:")
display(
    forecast_with_history.groupby("split").agg(
        total_examples=("significant_bloom", "size"),
        with_previous_CI=("has_previous_CI", "sum"),
    ).reindex(["train", "validation", "test"])
)

print("\nExamples with previous CI:")
display(
    forecast_with_history.loc[
        forecast_with_history["has_previous_CI"],
        [
            "period_start",
            "period_end",
            "previous_period_end",
            "previous_CI",
            "previous_bloom_class",
            "significant_bloom",
        ],
    ].head(10)
)

Previous CI availability:


,total_examples,with_previous_CI
split,,
train,187,171
validation,22,20
test,31,28



Examples with previous CI:


,period_start,period_end,previous_period_end,previous_CI,previous_bloom_class,significant_bloom
1,2002-06-11,2002-06-20,2002-06-10,1.649855,0.0,0
2,2002-06-21,2002-06-30,2002-06-20,1.722002,0.0,0
3,2002-07-01,2002-07-10,2002-06-30,2.208485,0.0,0
4,2002-07-11,2002-07-20,2002-07-10,2.108872,0.0,0
5,2002-07-21,2002-07-30,2002-07-20,2.702665,0.0,0
6,2002-07-31,2002-08-09,2002-07-30,2.539468,0.0,0
7,2002-08-10,2002-08-19,2002-08-09,2.791853,0.0,0
8,2002-08-20,2002-08-29,2002-08-19,2.573102,0.0,0
9,2002-08-30,2002-09-08,2002-08-29,3.150023,0.0,0
10,2002-09-09,2002-09-18,2002-09-08,2.876797,0.0,0


In [12]:
from sklearn.base import clone

# Use only examples with an adjacent previous CI observation.
history_train = forecast_with_history.loc[
    forecast_with_history["split"].eq("train")
    & forecast_with_history["has_previous_CI"]
].copy()

history_validation = forecast_with_history.loc[
    forecast_with_history["split"].eq("validation")
    & forecast_with_history["has_previous_CI"]
].copy()

features_with_CI = feature_columns + ["previous_CI"]

y_history_train = history_train["significant_bloom"]
y_history_validation = history_validation["significant_bloom"]

# Clone preserves the agreed settings but creates untrained models.
rf_environment_matched = clone(rf_classifier)
rf_with_CI = clone(rf_classifier)

rf_environment_matched.fit(
    history_train[feature_columns],
    y_history_train,
)

rf_with_CI.fit(
    history_train[features_with_CI],
    y_history_train,
)

def predict_at_half(model, inputs):
    positive_index = list(model.classes_).index(1)
    scores = model.predict_proba(inputs)[:, positive_index]
    return (scores >= 0.5).astype(int)

history_predictions = {
    "Persistence": (
        history_validation["previous_bloom_class"]
        .astype(int)
        .to_numpy()
    ),
    "Environmental RF": predict_at_half(
        rf_environment_matched,
        history_validation[feature_columns],
    ),
    "Environmental RF + previous CI": predict_at_half(
        rf_with_CI,
        history_validation[features_with_CI],
    ),
}

comparison_rows = []

for name, predictions in history_predictions.items():
    tn, fp, fn, tp = confusion_matrix(
        y_history_validation, predictions, labels=[0, 1]
    ).ravel()

    comparison_rows.append({
        "approach": name,
        "examples": len(y_history_validation),
        "accuracy": accuracy_score(y_history_validation, predictions),
        "precision": precision_score(
            y_history_validation, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_history_validation, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_history_validation, predictions, zero_division=0
        ),
        "correct_warnings": tp,
        "false_alarms": fp,
        "missed_significant": fn,
        "correct_lower_severity": tn,
    })

history_comparison = pd.DataFrame(comparison_rows)
display(history_comparison.round(3))

,approach,examples,accuracy,precision,recall,F1,correct_warnings,false_alarms,missed_significant,correct_lower_severity
0,Persistence,20,0.75,0.769,0.833,0.800,10,3,2,5
1,Environmental RF,20,0.65,0.632,1.000,0.774,12,7,0,1
2,Environmental RF + previous CI,20,0.75,0.706,1.000,0.828,12,5,0,3


In [13]:
# Combine training and validation data for the final fit.
development = forecast_with_history.loc[
    forecast_with_history["split"].isin(["train", "validation"])
    & forecast_with_history["has_previous_CI"]
].copy()

final_test = forecast_with_history.loc[
    forecast_with_history["split"].eq("test")
    & forecast_with_history["has_previous_CI"]
].copy()

# Clone retains the selected settings without the previous fitted trees.
final_classifier = clone(rf_with_CI)

final_classifier.fit(
    development[features_with_CI],
    development["significant_bloom"],
)

# Generate test predictions before comparing with actual outcomes.
positive_index = list(final_classifier.classes_).index(1)

final_test_scores = final_classifier.predict_proba(
    final_test[features_with_CI]
)[:, positive_index]

final_test_predictions = (final_test_scores >= 0.5).astype(int)

persistence_test_predictions = (
    final_test["previous_bloom_class"].astype(int).to_numpy()
)

y_final_test = final_test["significant_bloom"]

print("Development examples:", len(development))
print("Test examples:", len(final_test))

test_result_rows = []

for name, predictions in {
    "Persistence": persistence_test_predictions,
    "Environmental RF + previous CI": final_test_predictions,
}.items():
    tn, fp, fn, tp = confusion_matrix(
        y_final_test, predictions, labels=[0, 1]
    ).ravel()

    test_result_rows.append({
        "approach": name,
        "examples": len(y_final_test),
        "accuracy": accuracy_score(y_final_test, predictions),
        "precision": precision_score(
            y_final_test, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_final_test, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_final_test, predictions, zero_division=0
        ),
        "correct_warnings": tp,
        "false_alarms": fp,
        "missed_significant": fn,
        "correct_lower_severity": tn,
    })

final_classification_results = pd.DataFrame(test_result_rows)
display(final_classification_results.round(3))

# Retain individual predictions for later plots and error analysis.
final_classification_predictions = final_test[
    ["period_start", "period_end", "significant_bloom"]
].copy()

final_classification_predictions["model_score"] = final_test_scores
final_classification_predictions["model_prediction"] = final_test_predictions
final_classification_predictions["persistence_prediction"] = (
    persistence_test_predictions
)

Development examples: 191
Test examples: 28


,approach,examples,accuracy,precision,recall,F1,correct_warnings,false_alarms,missed_significant,correct_lower_severity
0,Persistence,28,0.786,0.700,0.7,0.700,7,3,3,15
1,Environmental RF + previous CI,28,0.714,0.562,0.9,0.692,9,7,1,11


In [14]:
from pathlib import Path

PROJECT = Path(
    "/home/samuel-logan/uni/ENGG2112/lake-erie-forecast"
)
PROCESSED = PROJECT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

environment_features = [
    "flow_mean_10d",
    "rainfall_total_10d",
    "maumee_level_mean_10d",
    "detroit_level_mean_10d",
    "day_of_year",
]

export_columns = [
    "period_start",
    "significant_bloom",
    *environment_features,
    "previous_CI",
]

environment_export = forecast_with_history[
    export_columns
].copy()

assert environment_export["period_start"].is_unique

environment_export.to_csv(
    PROCESSED / "erie_environment_features.csv",
    index=False,
)

print("Saved environmental examples:", len(environment_export))

Saved environmental examples: 240
